In [9]:
import ee
import numpy as np

# Initialize the Earth Engine module
ee.Initialize(project='usfs-carbon-viz-test')

def log_bins(start, stop, bins):
    return np.logspace(start=np.log10(start), stop=np.log10(stop), num=bins+1).tolist()

# Function to reclassify the image into logarithmic bins
def log_binning(image, bin_edges):
    # Create an empty image to hold the reclassified values
    reclassified = ee.Image(0)
    
    # Iterate over the bin edges and assign bin values
    for i in range(1, len(bin_edges)):
        lower = bin_edges[i - 1]
        upper = bin_edges[i]
        bin_value = i
        
        # Create a mask for the current bin
        mask = image.gte(lower).And(image.lt(upper))
        
        # Update the reclassified image with the bin value
        reclassified = reclassified.where(mask, bin_value)
    
    return reclassified.updateMask(image)

In [10]:
import json
with open('nested_data.json', 'r') as json_file:
    data_dict = json.load(json_file)

data_dict

{'Total Initial Forest Carbon 2014': {'AssetID': 'projects/usfs-carbon-viz-test/assets/Carbon/TM_2014_Carbon_TotalInitial',
  'Min': 0,
  'Max': 1190,
  'Bands': ['b1']},
 'Total Forest Carbon Emissions by Flame Length 2014': {'AssetID': 'projects/usfs-carbon-viz-test/assets/Carbon/Emissions/TM2014_Carbon_Emissions',
  'Min': 0,
  'Max': 61.5,
  'Bands': ['b1', 'b2', 'b3', 'b4', 'b5', 'b6']},
 'Expected Annual Forest Carbon Emissions 2014': {'AssetID': 'projects/usfs-carbon-viz-test/assets/Carbon/Emissions/TM2014_Carbon_Emissions',
  'Min': 0,
  'Max': 14,
  'Bands': ['b7']},
 'Total Forest Carbon Remaining by Flame Length 2014': {'AssetID': 'projects/usfs-carbon-viz-test/assets/Carbon/Remaining/TM2014_Carbon_Remaining',
  'Min': 0,
  'Max': 1172,
  'Bands': ['b1', 'b2', 'b3', 'b4', 'b5', 'b6']},
 'Expected Annual Forest Carbon Remaining 2014': {'AssetID': 'projects/usfs-carbon-viz-test/assets/Carbon/Remaining/TM2014_Carbon_Remaining',
  'Min': 0,
  'Max': 1190,
  'Bands': ['b7']},
 'T

In [11]:
coords =   [
    [
      [
        -123.02547024320674,
        43.586180417918825
      ],
      [
        -122.89844082426143,
        43.586180417918825
      ],
      [
        -122.89844082426143,
        43.646331323686304
      ],
      [
        -123.02547024320674,
        43.646331323686304
      ],
      [
        -123.02547024320674,
        43.586180417918825
      ]
    ]
  ]
geom = ee.Geometry.Polygon(coords=coords)
print(geom.getInfo())

coords2 = [[[-122.563477, 36.879621],
   [-122.563477, 37.770715],
   [-119.421387, 37.770715],
   [-119.421387, 36.879621],
   [-122.563477, 36.879621]]]
geom2 = ee.Geometry.Polygon(coords=coords2)
print(geom2.getInfo())

{'type': 'Polygon', 'coordinates': [[[-123.02547024320674, 43.586180417918825], [-122.89844082426143, 43.586180417918825], [-122.89844082426143, 43.646331323686304], [-123.02547024320674, 43.646331323686304], [-123.02547024320674, 43.586180417918825]]]}
{'type': 'Polygon', 'coordinates': [[[-122.563477, 36.879621], [-119.421387, 36.879621], [-119.421387, 37.770715], [-122.563477, 37.770715], [-122.563477, 36.879621]]]}


In [12]:
l = list(data_dict.keys())
for li in l:
    print(li)

Total Initial Forest Carbon 2014
Total Forest Carbon Emissions by Flame Length 2014
Expected Annual Forest Carbon Emissions 2014
Total Forest Carbon Remaining by Flame Length 2014
Expected Annual Forest Carbon Remaining 2014
Total Initial Standing Live and Dead Tree Carbon 2014
Carbon Loss from Standing Live and Dead Trees by Flame Length 2014
Expected Annual Carbon Loss from Standing Live and Dead Trees 2014
Remaining Carbon in Standing Live and Dead Trees by Flame Length 2014
Expected Annual Carbon Remaining in Standing Live and Dead Trees 2014
Total Initial Rangeland Carbon 2014
Total Conditional Rangeland Carbon Remaining 2014
Total Conditional Rangeland Carbon Emission 2014
Total Annual Expected Rangeland Carbon 2014
Total Annual Expected Rangeland Carbon Emissions 2014
Burn Probability 2014
Flame Length Probability FSim 2014
Projected Burn Probability circa 2047
Projected Flame Length Probability circa 2047


## QA a particular layer

In [13]:
k = "Total Initial Forest Carbon 2014"
v = data_dict[k]
print(v)

print(f"Min: {v['Min']}, Max: {v['Max']}")
img = ee.Image(v['AssetID'])

# 'correct' using 0 as start of logbin edges
placeholder_min = 1e-3 # avoid divide by zero nan's
logbin = log_bins(placeholder_min, v['Max'], 18)
logbin[0] = 0 # replace placeholder back to zero
print(f"logbin_right: {logbin}")

# 'wrong' using modified min as start of logbin edges
# if v['Max'] > 40:
#     fake_min = 1
# else:
#     fake_min = 1e-3
logbin_wrong = log_bins(1, v['Max'], 18)
print(f"logbin_wrong: {logbin_wrong}")
proj = img.projection().getInfo()
# print(proj.keys())
out_assetID = f"{v['AssetID']}_BINNED_test_fix_geom2"

bandnames = v["Bands"]
crs = proj['wkt'] if 'wkt' in proj.keys() else proj['crs']

if len(bandnames) > 1:
    print(f"More than one band: {bandnames}")
    img_list_right = []
    img_list_wrong = []
    for b in v['Bands']:
        print(b)
        logbinned_right = log_binning(img.select(b), logbin)
        logbinned_wrong = log_binning(img.select(b), logbin_wrong)

        img_list_right.append(logbinned_right)
        img_list_wrong.append(logbinned_wrong)
    out_img_right = ee.Image.cat(img_list_right).rename(bandnames)
    out_img_wrong = ee.Image.cat(img_list_wrong).rename(bandnames)
else:
    # usfs folks stored some of these distinct layers as the 7th band in another dataset i.e. emissions by flame lengths 1-6 + 7th band is carbon remaining
    if bandnames == ['b7']:
        print('This layer uses band 7 of another image, appending assetID..')
        out_assetID = out_assetID + "_B7"
        img = img.select('b7')
    
    out_img_right = log_binning(img, logbin).rename(bandnames)
    out_img_wrong = log_binning(img, logbin_wrong).rename(bandnames)
        


{'AssetID': 'projects/usfs-carbon-viz-test/assets/Carbon/TM_2014_Carbon_TotalInitial', 'Min': 0, 'Max': 1190, 'Bands': ['b1']}
Min: 0, Max: 1190
logbin_right: [0, 0.0021753562341587235, 0.0047321747454932275, 0.010294165833737162, 0.022393477821883875, 0.04871379158433022, 0.10596985021248151, 0.23052217429258798, 0.501467848959205, 1.0908712114635717, 2.3730334905215624, 5.162193197473516, 11.229609154055767, 24.428400280441103, 53.14047284058231, 115.59945887990294, 251.4700035397721, 547.03683990416, 1189.9999999999995]
logbin_wrong: [1.0, 1.4820529497081318, 2.1964809457385748, 3.2553010646095615, 4.824528544992623, 7.1502067610574, 10.596985021248145, 15.705292908753705, 23.27607568144864, 34.49637662132067, 51.12545672587095, 75.7706339457525, 112.29609154055755, 166.42875370837774, 246.65622534974943, 365.5575863434699, 541.7756991285247, 802.9402729736155, 1189.9999999999995]


In [14]:
import geemap 
Map = geemap.Map()
emissions_pal = ['ffffcc', 'fbec9a', 'f4cc68', 'eca855', 'e48751', 'd2624d', 'a54742', '73382f', '422818', '1a1a01']
carbon_pal = ['3e1f0d', '5a2e15', '783f1f', '97542b', 'b66c3b', 'd1864e', 'e6a062', 'f2ba7a', 'f9d395', 'e1e9a5', 'b4da90', '84c87c', '53b069', '2b8c52', '0a643a', '166e5c', '105a80', '0a449e']
Map.addLayer(img.select(0), {'min':v["Min"], 'max': v["Max"], 'palette': carbon_pal}, 'Original Image')
Map.addLayer(img.select(0), {'min':v["Min"],'max': 950, 'palette': carbon_pal}, '90th pct cutoff')
Map.addLayer(out_img_right.select(0), {'min': 1, 'max': 18, 'palette': carbon_pal},'log binned 18 bins right')
Map.addLayer(out_img_wrong.select(0), {'min': 1, 'max': 18, 'palette': carbon_pal},'log binned 18 bins wrong')
# Map.addLayer(out_img_right.select(1), {'min': 1, 'max': 18, 'palette': ['3e1f0d', '5a2e15', '783f1f', '97542b', 'b66c3b', 'd1864e', 'e6a062', 'f2ba7a', 'f9d395', 'e1e9a5', 'b4da90', '84c87c', '53b069', '2b8c52', '0a643a', '166e5c', '105a80', '0a449e']},'log binned 18 bins right FL2')
# Map.addLayer(out_img_wrong.select(1), {'min': 1, 'max': 18, 'palette': ['3e1f0d', '5a2e15', '783f1f', '97542b', 'b66c3b', 'd1864e', 'e6a062', 'f2ba7a', 'f9d395', 'e1e9a5', 'b4da90', '84c87c', '53b069', '2b8c52', '0a643a', '166e5c', '105a80', '0a449e']},'log binned 18 bins wrong FL2')

# Map.addLayer(img.geometry())
# Map.addLayer(geom)
Map.centerObject(img, 2)
Map

Map(center=[38.516429089005484, -96.56365546520736], controls=(WidgetControl(options=['position', 'transparent…

## Automation

In [16]:
# automated exports
from pathlib import Path
select_layers = [
    # "Total Initial Forest Carbon 2014",
    "Total Forest Carbon Remaining by Flame Length 2014",
    "Expected Annual Forest Carbon Remaining 2014",
    "Total Initial Standing Live and Dead Tree Carbon 2014",
    "Remaining Carbon in Standing Live and Dead Trees by Flame Length 2014",
    "Expected Annual Carbon Remaining in Standing Live and Dead Trees 2014"
]
for k,v in data_dict.items():
    if k not in select_layers: 
        print(f'skipping {k}')
        continue
    print(k)
    print(f"Min: {v['Min']}, Max: {v['Max']}")
    img = ee.Image(v['AssetID'])

    # 'correct' using 0 as start of logbin edges, looks terrible bc too many bins inside the 0-1 range
    placeholder_min = 1e-3 # avoid divide by zero nan's
    logbin = log_bins(placeholder_min, v['Max'], 18)
    logbin[0] = 0 # replace placeholder back to zero
    print(f"logbin_right: {logbin}")
    
    # 'wrong' using modified min as start of logbin edges looks better 
    logbin_wrong = log_bins(1, v['Max'], 18)
    print(f"logbin_wrong: {logbin_wrong}")
    proj = img.projection().getInfo()
    print(proj.keys())
    out_assetID = f"{v['AssetID']}_BINNED_2"
    
    bandnames = v["Bands"]
    crs = proj['wkt'] if 'wkt' in proj.keys() else proj['crs']
    
    if len(bandnames) > 1:
        print(f"More than one band: {bandnames}")
        img_list_right = []
        img_list_wrong = []
        for b in v['Bands']:
            print(b)
            logbinned_right = log_binning(img.select(b), logbin)
            logbinned_wrong = log_binning(img.select(b), logbin_wrong)

            img_list_right.append(logbinned_right)
            img_list_wrong.append(logbinned_wrong)
        out_img_right = ee.Image.cat(img_list_right).rename(bandnames)
        out_img_wrong = ee.Image.cat(img_list_wrong).rename(bandnames)
        
        task = ee.batch.Export.image.toAsset(
            image=out_img_wrong,
            description=Path(out_assetID).stem,
            assetId=out_assetID,
            scale=30,
            crs=crs,
            crsTransform=proj['transform'],
            region=img.geometry(), #geom2, #img.geometry()
            pyramidingPolicy={'.default': 'mode'},
            maxPixels=1e13
        )
        print(f"Exporting {out_assetID}\n")
        task.start()
    
    else:
        # usfs folks stored some of these distinct layers as the 7th band in another dataset i.e. emissions by flame lengths 1-6 + 7th band is carbon remaining
        if bandnames == ['b7']:
            print('This layer uses band 7 of another image, appending assetID..')
            out_assetID = out_assetID + "_B7"
            img = img.select('b7')
        
        out_img_right = log_binning(img, logbin).rename(bandnames)
        out_img_wrong = log_binning(img, logbin_wrong).rename(bandnames)
        
        task = ee.batch.Export.image.toAsset(
            image=out_img_wrong,
            description=Path(out_assetID).stem,
            assetId=out_assetID,
            scale=30,
            crs=crs,
            crsTransform=proj['transform'],
            region=img.geometry(), #geom2, #img.geometry()            
            pyramidingPolicy={'.default': 'mode'},
            maxPixels=1e13

        )
        print(f"Exporting {out_assetID}\n")
        task.start()



skipping Total Initial Forest Carbon 2014
skipping Total Forest Carbon Emissions by Flame Length 2014
skipping Expected Annual Forest Carbon Emissions 2014
Total Forest Carbon Remaining by Flame Length 2014
Min: 0, Max: 1172
logbin_right: [0, 0.002173515016893582, 0.004724167528661913, 0.010268049065867715, 0.02231775883886361, 0.04850798397967956, 0.10543283161906687, 0.22915984279765458, 0.4980823595896748, 1.0825894882179472, 2.3530245097728484, 5.114334107109951, 11.116081983214508, 24.160971119536924, 52.514233551045756, 114.14047522385474, 248.0860369344181, 539.2187267585737, 1171.9999999999993]
logbin_wrong: [1.0, 1.480798542987979, 2.1927643249153217, 3.247042217450628, 4.808215384621347, 7.119998335919677, 10.543283161906693, 15.61247834446113, 23.11893518490941, 34.23448553724737, 50.69437630349896, 75.06815856790561, 111.1608198321452, 164.60678004478982, 243.74948005626769, 360.9438749213986, 534.4851640840423, 791.4648522243406, 1171.9999999999993]
dict_keys(['type', 'wkt